In [0]:
# ===== Phase 1: Data Understanding =====
import pandas as pd
import re
from sklearn.preprocessing import MultiLabelBinarizer
from collections import Counter

# Load the dataset
df = pd.read_csv("/Volumes/workspace/default/anime_data/top_15000_anime.csv")
print("Raw shape:", df.shape)

# Check missing values
print("\nMissing values:\n", df.isnull().sum())

# Drop rows with no genre label
df = df.dropna(subset=["genres"]).reset_index(drop=True)
print("\nShape after dropping missing genres:", df.shape)

# Explore the multi-label genre structure
genre_lists = df["genres"].apply(lambda x: [g.strip() for g in x.split(",")])

all_genres = set(g for sub in genre_lists for g in sub)
print("\nNumber of unique genres:", len(all_genres))
print(sorted(all_genres))

genres_per_anime = genre_lists.apply(len)
print("\nGenres per anime -> min:", genres_per_anime.min(),
      "max:", genres_per_anime.max(),
      "avg:", round(genres_per_anime.mean(), 2))

# Check genre frequency (imbalance)
flat = [g for sub in genre_lists for g in sub]
genre_counts = pd.Series(Counter(flat)).sort_values(ascending=False)
print("\nGenre frequency:\n", genre_counts)

# Clean and engineer input features (X)
df["episodes"] = df["episodes"].fillna(df["episodes"].median())
df["rating"] = df["rating"].fillna("Unknown")
df["studios"] = df["studios"].fillna("Unknown")

def parse_duration(d):
    match = re.search(r"(\d+)", str(d))
    return int(match.group(1)) if match else 0

df["duration_min"] = df["duration"].apply(parse_duration)

top_studios = df["studios"].value_counts().nlargest(20).index
df["studio_grouped"] = df["studios"].apply(
    lambda s: s if s in top_studios else "Other"
)

feature_cols = [
    "score", "type", "episodes", "source", "duration_min",
    "rating", "popularity", "members", "favorites",
    "scored_by", "studio_grouped",
]
features = df[feature_cols].copy()
print("\nMissing values in features:\n", features.isnull().sum())

# Build the multi-label target matrix (y)
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(genre_lists)
genre_df = pd.DataFrame(genre_matrix, columns=mlb.classes_)
print("\nTarget matrix shape:", genre_df.shape)

# Combine features + target, save clean dataset
clean_df = pd.concat([features, genre_df], axis=1)
clean_df.to_csv("/Volumes/workspace/default/anime_data/anime_clean.csv", index=False)

print("\nSaved anime_clean.csv, shape:", clean_df.shape)
clean_df.head()

Raw shape: (15000, 24)

Missing values:
 anime_id              0
anime_url             0
image_url             0
name                  0
english_name       6252
japanese_names       34
score                 0
genres              618
themes             5834
demographics      10381
synopsis            514
type                  0
episodes            111
premiered          9988
producers          5070
studios            1754
source                0
duration              0
rating               79
rank               1521
popularity            0
favorites             0
scored_by             0
members               0
dtype: int64

Shape after dropping missing genres: (14382, 24)

Number of unique genres: 21
['Action', 'Adventure', 'Avant Garde', 'Award Winning', 'Boys Love', 'Comedy', 'Drama', 'Ecchi', 'Erotica', 'Fantasy', 'Girls Love', 'Gourmet', 'Hentai', 'Horror', 'Mystery', 'Romance', 'Sci-Fi', 'Slice of Life', 'Sports', 'Supernatural', 'Suspense']

Genres per anime -> min: 1 max: 7 avg: 

,score,type,episodes,source,duration_min,rating,popularity,members,favorites,scored_by,studio_grouped,Action,Adventure,Avant Garde,Award Winning,Boys Love,Comedy,Drama,Ecchi,Erotica,Fantasy,Girls Love,Gourmet,Hentai,Horror,Mystery,Romance,Sci-Fi,Slice of Life,Sports,Supernatural,Suspense
0,9.29,TV,28.0,Manga,24,PG-13 - Teens 13 or older,128,1225468,76513,734207.0,Madhouse,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0
1,9.10,TV,64.0,Manga,24,R - 17+ (violence & profanity),3,3577489,236798,2249670.0,Bones,1,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0
2,9.07,TV,24.0,Visual novel,24,PG-13 - Teens 13 or older,14,2737980,198296,1483605.0,Other,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1
3,9.05,TV,10.0,Manga,23,R - 17+ (violence & profanity),21,2497671,61832,1729484.0,Other,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
4,9.05,TV,51.0,Manga,24,PG-13 - Teens 13 or older,344,676352,17315,266825.0,Other,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0


In [0]:
# ===== Phase 2: Baseline Model =====
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
import mlflow
import mlflow.sklearn

# Load the clean dataset from Phase 1
df = pd.read_csv("/Volumes/workspace/default/anime_data/anime_clean.csv")
print("Loaded shape:", df.shape)

genre_cols = [
    "Action", "Adventure", "Avant Garde", "Award Winning", "Boys Love",
    "Comedy", "Drama", "Ecchi", "Erotica", "Fantasy", "Girls Love",
    "Gourmet", "Hentai", "Horror", "Mystery", "Romance", "Sci-Fi",
    "Slice of Life", "Sports", "Supernatural", "Suspense",
]

X = df.drop(columns=genre_cols)
y = df[genre_cols]

# One-hot encode categorical columns
categorical_cols = ["type", "source", "rating", "studio_grouped"]
X = pd.get_dummies(X, columns=categorical_cols)
print("Feature matrix shape after encoding:", X.shape)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

# Train baseline model + log to MLflow
mlflow.set_experiment("/Users/" + spark.sql("SELECT current_user()").collect()[0][0] + "/Anime_Genre_Prediction")

with mlflow.start_run(run_name="baseline_logistic_regression"):
    model = OneVsRestClassifier(
        LogisticRegression(max_iter=1000, random_state=42)
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    f1_micro = f1_score(y_test, y_pred, average="micro", zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
    precision_micro = precision_score(y_test, y_pred, average="micro", zero_division=0)
    recall_micro = recall_score(y_test, y_pred, average="micro", zero_division=0)

    print(f"F1 (micro): {f1_micro:.4f}")
    print(f"F1 (macro): {f1_macro:.4f}")
    print(f"Precision (micro): {precision_micro:.4f}")
    print(f"Recall (micro): {recall_micro:.4f}")

    mlflow.log_param("model", "LogisticRegression (OneVsRest)")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("test_size", 0.2)

    mlflow.log_metric("f1_micro", f1_micro)
    mlflow.log_metric("f1_macro", f1_macro)
    mlflow.log_metric("precision_micro", precision_micro)
    mlflow.log_metric("recall_micro", recall_micro)

    mlflow.sklearn.log_model(model, "model")

    print("\nRun logged to MLflow. Check the Experiments tab in the sidebar.")

Loaded shape: (14382, 32)
Feature matrix shape after encoding: (14382, 58)
Train shape: (11505, 58) Test shape: (2877, 58)


2026/07/27 10:26:08 INFO mlflow.tracking.fluent: Experiment with name '/Users/sabhareeshbalaji0307@gmail.com/Anime_Genre_Prediction' does not exist. Creating a new experiment.
/databricks/python/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/databricks/python/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/mod

In [0]:
# ===== Phase 3: Better Models =====
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
import mlflow
import mlflow.sklearn

# Load data (same prep as Phase 2)
df = pd.read_csv("/Volumes/workspace/default/anime_data/anime_clean.csv")

genre_cols = [
    "Action", "Adventure", "Avant Garde", "Award Winning", "Boys Love",
    "Comedy", "Drama", "Ecchi", "Erotica", "Fantasy", "Girls Love",
    "Gourmet", "Hentai", "Horror", "Mystery", "Romance", "Sci-Fi",
    "Slice of Life", "Sports", "Supernatural", "Suspense",
]

X = df.drop(columns=genre_cols)
y = df[genre_cols]

categorical_cols = ["type", "source", "rating", "studio_grouped"]
X = pd.get_dummies(X, columns=categorical_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Define the models we want to compare
models = {
    "random_forest": RandomForestClassifier(
        n_estimators=200, max_depth=15, random_state=42, n_jobs=-1
    ),
    "gradient_boosting": GradientBoostingClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42
    ),
}

mlflow.set_experiment("/Users/" + spark.sql("SELECT current_user()").collect()[0][0] + "/Anime_Genre_Prediction")

# Train each model, log to MLflow
for model_name, base_model in models.items():
    with mlflow.start_run(run_name=model_name):
        print(f"\nTraining {model_name}...")

        model = OneVsRestClassifier(base_model, n_jobs=-1)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        f1_micro = f1_score(y_test, y_pred, average="micro", zero_division=0)
        f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
        precision_micro = precision_score(y_test, y_pred, average="micro", zero_division=0)
        recall_micro = recall_score(y_test, y_pred, average="micro", zero_division=0)

        print(f"{model_name} -> F1 micro: {f1_micro:.4f}, F1 macro: {f1_macro:.4f}")

        mlflow.log_param("model", model_name)
        for param_name, param_value in base_model.get_params().items():
            mlflow.log_param(param_name, param_value)

        mlflow.log_metric("f1_micro", f1_micro)
        mlflow.log_metric("f1_macro", f1_macro)
        mlflow.log_metric("precision_micro", precision_micro)
        mlflow.log_metric("recall_micro", recall_micro)

        mlflow.sklearn.log_model(model, "model")

print("\nAll runs logged. Check the Experiments tab to compare all 3 runs.")


Training random_forest...
random_forest -> F1 micro: 0.4254, F1 macro: 0.2548


2026/07/27 10:37:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-d09134e0-e9de.cloud.databricks.com/ml/experiments/1206966696393049/models/m-d8be91ecaaff448b92cb5a04f183b21f?o=7474649036062716
2026/07/27 10:37:50 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.



Training gradient_boosting...
gradient_boosting -> F1 micro: 0.4457, F1 macro: 0.3133


2026/07/27 10:38:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-d09134e0-e9de.cloud.databricks.com/ml/experiments/1206966696393049/models/m-c1c4f64d039d4a1b85c98fc0f2908519?o=7474649036062716
2026/07/27 10:38:57 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.



All runs logged. Check the Experiments tab to compare all 3 runs.


In [0]:
# ===== Phase 4: Model Registration =====
import mlflow
from mlflow import MlflowClient
from mlflow.models import infer_signature
import pandas as pd

# Set the registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

MODEL_NAME = "workspace.default.anime_genre_predictor"

client = MlflowClient()

# Get current user's experiment path (same as Phases 2-3)
username = spark.sql("SELECT current_user()").collect()[0][0]
experiment_path = f"/Users/{username}/Anime_Genre_Prediction"
experiment = client.get_experiment_by_name(experiment_path)

# Find the best run by f1_micro
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1_micro DESC"],
    max_results=1,
)

best_run = runs[0]
print(f"Best run: {best_run.data.tags.get('mlflow.runName')}")
print(f"  f1_micro: {best_run.data.metrics['f1_micro']:.4f}")
print(f"  f1_macro: {best_run.data.metrics['f1_macro']:.4f}")
print(f"  Run ID: {best_run.info.run_id}")

# Rebuild a small sample input + prediction to create a signature
# (Unity Catalog requires this, unlike local MLflow)
df = pd.read_csv("/Volumes/workspace/default/anime_data/anime_clean.csv")
genre_cols = [
    "Action", "Adventure", "Avant Garde", "Award Winning", "Boys Love",
    "Comedy", "Drama", "Ecchi", "Erotica", "Fantasy", "Girls Love",
    "Gourmet", "Hentai", "Horror", "Mystery", "Romance", "Sci-Fi",
    "Slice of Life", "Sports", "Supernatural", "Suspense",
]
X = df.drop(columns=genre_cols)
categorical_cols = ["type", "source", "rating", "studio_grouped"]
X = pd.get_dummies(X, columns=categorical_cols)
sample_input = X.iloc[:5]

loaded_model = mlflow.pyfunc.load_model(f"runs:/{best_run.info.run_id}/model")
sample_output = loaded_model.predict(sample_input)
signature = infer_signature(sample_input, sample_output)

# Re-log the model WITH a signature under a new run, then register that
with mlflow.start_run(run_name="gradient_boosting_with_signature"):
    mlflow.sklearn.log_model(
        loaded_model._model_impl.sklearn_model,
        "model",
        signature=signature,
        input_example=sample_input,
    )
    new_run_id = mlflow.active_run().info.run_id

model_uri = f"runs:/{new_run_id}/model"
registered_model = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)
print(f"\nRegistered as '{MODEL_NAME}', version {registered_model.version}")

# Set alias to "production"
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="production",
    version=registered_model.version,
)

client.update_model_version(
    name=MODEL_NAME,
    version=registered_model.version,
    description=(
        f"Gradient Boosting, chosen as the best model out of "
        f"Logistic Regression, Random Forest, and Gradient Boosting. "
        f"It had the highest f1_micro ({best_run.data.metrics['f1_micro']:.4f}) "
        f"and f1_macro ({best_run.data.metrics['f1_macro']:.4f})."
    ),
)

print(f"\n'{MODEL_NAME}' version {registered_model.version} is now aliased as 'production'.")
print("Check the 'Models' tab in the left sidebar.")

Best run: gradient_boosting
  f1_micro: 0.4457
  f1_macro: 0.3133
  Run ID: 73c5edd344394ef486c3b4dcd6595b48


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/07/27 10:42:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-d09134e0-e9de.cloud.databricks.com/ml/experiments/1206966696393049/models/m-3eb0b04bacd94e6292d6a

Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.anime_genre_predictor': https://dbc-d09134e0-e9de.cloud.databricks.com/explore/data/models/workspace/default/anime_genre_predictor/version/1?o=7474649036062716



Registered as 'workspace.default.anime_genre_predictor', version 1

'workspace.default.anime_genre_predictor' version 1 is now aliased as 'production'.
Check the 'Models' tab in the left sidebar.


In [0]:
# ===== Dynamic Genre Predictor (Databricks)  =====
import mlflow
import pandas as pd

mlflow.set_registry_uri("databricks-uc")
MODEL_NAME = "workspace.default.anime_genre_predictor"
ALIAS = "production"

genre_cols = [
    "Action", "Adventure", "Avant Garde", "Award Winning", "Boys Love",
    "Comedy", "Drama", "Ecchi", "Erotica", "Fantasy", "Girls Love",
    "Gourmet", "Hentai", "Horror", "Mystery", "Romance", "Sci-Fi",
    "Slice of Life", "Sports", "Supernatural", "Suspense",
]

# Rebuild the same one-hot encoded columns the model expects
df = pd.read_csv("/Volumes/workspace/default/anime_data/anime_clean.csv")
X_reference = df.drop(columns=genre_cols)
categorical_cols = ["type", "source", "rating", "studio_grouped"]
X_reference = pd.get_dummies(X_reference, columns=categorical_cols)
expected_columns = X_reference.columns.tolist()
expected_dtypes = X_reference.dtypes  # <-- capture the exact training dtypes

# ---- EDIT THESE VALUES to describe your own made-up anime ----
my_anime = {
    "score": 8.6,
    "episodes": 22,
    "type": "TV",
    "source": "Manga",
    "duration_min": 24,
    "rating": "PG-13 - Teens 13 or older",
    "popularity": 301,
    "members": 500000,
    "favorites": 8000,
    "scored_by": 150000,
    "studio_grouped": "Madhouse",
}
# ----------------------------------------------------------------

raw_input = pd.DataFrame([my_anime])
raw_input_encoded = pd.get_dummies(raw_input, columns=categorical_cols)

# Add any missing columns with 0
for col in expected_columns:
    if col not in raw_input_encoded.columns:
        raw_input_encoded[col] = 0
raw_input_encoded = raw_input_encoded[expected_columns]

# Force matching dtypes exactly (this fixes the schema error)
raw_input_encoded = raw_input_encoded.astype(expected_dtypes)

# Load the production model and predict
model_uri = f"models:/{MODEL_NAME}@{ALIAS}"
model = mlflow.pyfunc.load_model(model_uri)

prediction = model.predict(raw_input_encoded)
predicted_genres = [
    genre for genre, flag in zip(genre_cols, prediction[0]) if flag == 1
]

print("Your made-up anime:", my_anime)
print("\nPredicted genres:", predicted_genres if predicted_genres else "(none confidently predicted)")

Your made-up anime: {'score': 8.6, 'episodes': 22, 'type': 'TV', 'source': 'Manga', 'duration_min': 24, 'rating': 'PG-13 - Teens 13 or older', 'popularity': 301, 'members': 500000, 'favorites': 8000, 'scored_by': 150000, 'studio_grouped': 'Madhouse'}

Predicted genres: ['Drama']


In [0]:
# ===== Phase 6: Automatic Model Replacement =====
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score
from mlflow.models import infer_signature
import mlflow
from mlflow import MlflowClient

mlflow.set_registry_uri("databricks-uc")
MODEL_NAME = "workspace.default.anime_genre_predictor"
ALIAS = "production"

genre_cols = [
    "Action", "Adventure", "Avant Garde", "Award Winning", "Boys Love",
    "Comedy", "Drama", "Ecchi", "Erotica", "Fantasy", "Girls Love",
    "Gourmet", "Hentai", "Horror", "Mystery", "Romance", "Sci-Fi",
    "Slice of Life", "Sports", "Supernatural", "Suspense",
]

# Same data prep as before
df = pd.read_csv("/Volumes/workspace/default/anime_data/anime_clean.csv")
X = df.drop(columns=genre_cols)
y = df[genre_cols]
categorical_cols = ["type", "source", "rating", "studio_grouped"]
X = pd.get_dummies(X, columns=categorical_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train an IMPROVED Gradient Boosting model (more trees, lower learning rate)
improved_model = GradientBoostingClassifier(
    n_estimators=250, max_depth=5, learning_rate=0.05, random_state=42
)

username = spark.sql("SELECT current_user()").collect()[0][0]
experiment_path = f"/Users/{username}/Anime_Genre_Prediction"
mlflow.set_experiment(experiment_path)

with mlflow.start_run(run_name="gradient_boosting_v2_improved"):
    model = OneVsRestClassifier(improved_model, n_jobs=-1)
    print("Training improved model...")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    new_f1_micro = f1_score(y_test, y_pred, average="micro", zero_division=0)
    new_f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
    print(f"New model -> F1 micro: {new_f1_micro:.4f}, F1 macro: {new_f1_macro:.4f}")

    mlflow.log_param("model", "gradient_boosting_v2_improved")
    for param_name, param_value in improved_model.get_params().items():
        mlflow.log_param(param_name, param_value)
    mlflow.log_metric("f1_micro", new_f1_micro)
    mlflow.log_metric("f1_macro", new_f1_macro)

    # Build a signature (Unity Catalog requires it)
    sample_input = X_train.iloc[:5]
    sample_output = model.predict(sample_input)
    signature = infer_signature(sample_input, sample_output)

    mlflow.sklearn.log_model(model, "model", signature=signature, input_example=sample_input)
    new_run_id = mlflow.active_run().info.run_id

# Register this new run as a new model version
client = MlflowClient()
new_version = mlflow.register_model(model_uri=f"runs:/{new_run_id}/model", name=MODEL_NAME)
print(f"\nRegistered as version {new_version.version}")

# Get the CURRENT production version
current_prod = client.get_model_version_by_alias(MODEL_NAME, ALIAS)

# The production run might not have f1_micro logged directly (it was
# re-logged just for its signature in Phase 4). So instead, look up the
# ORIGINAL "gradient_boosting" run in the experiment to get its real f1_micro.
experiment = client.get_experiment_by_name(experiment_path)
original_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName = 'gradient_boosting'",
    order_by=["metrics.f1_micro DESC"],
    max_results=1,
)
current_f1_micro = original_runs[0].data.metrics["f1_micro"]

print(f"\nCurrent production version: {current_prod.version} (f1_micro={current_f1_micro:.4f})")
print(f"New candidate version:      {new_version.version} (f1_micro={new_f1_micro:.4f})")

# Promote automatically ONLY if better
if new_f1_micro > current_f1_micro:
    client.set_registered_model_alias(name=MODEL_NAME, alias="archived", version=current_prod.version)
    client.set_registered_model_alias(name=MODEL_NAME, alias=ALIAS, version=new_version.version)
    print(f"\nPROMOTED: version {new_version.version} is now 'production' (version {current_prod.version} archived).")
else:
    print(f"\nNOT PROMOTED: version {new_version.version} did not beat production. Still version {current_prod.version}.")

Training improved model...
New model -> F1 micro: 0.4563, F1 macro: 0.3221


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/07/27 11:09:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-d09134e0-e9de.cloud.databricks.com/ml/experiments/1206966696393049/models/m-52a30e21d9614a06afe99

Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '4' of model 'workspace.default.anime_genre_predictor': https://dbc-d09134e0-e9de.cloud.databricks.com/explore/data/models/workspace/default/anime_genre_predictor/version/4?o=7474649036062716



Registered as version 4

Current production version: 1 (f1_micro=0.4457)
New candidate version:      4 (f1_micro=0.4563)

PROMOTED: version 4 is now 'production' (version 1 archived).
